In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

import json
import os
from pathlib import Path
from time import perf_counter
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import Markdown, display

def find_skill_dir():
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / "data_platform/mcp_server/skills"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy data_platform/mcp_server/skills")


def load_skill_registry():
    registry = {}
    skill_dir = find_skill_dir()
    for skill_file in sorted(skill_dir.glob("*/*.md")):
        if skill_file.name.lower() != "skill.md":
            continue
        content = skill_file.read_text(encoding="utf-8")
        _, front_matter, instructions = content.split("---", 2)
        metadata = dict(
            line.split(":", 1) for line in front_matter.strip().splitlines()
            if ":" in line
        )
        name = metadata["name"].strip()
        registry[name] = {
            "description": metadata["description"].strip(),
            "instructions": instructions.strip(),
        }
    return registry


async def main(user_question="Which product sold the most units during this calendar week? Show the product name and total units sold in a Markdown table."):
    load_dotenv(find_skill_dir().parents[2] / ".env")
    client = MultiServerMCPClient(
        {
            "bakery_cube": {
                "transport": "http",
                "url": f"http://localhost:{os.getenv('MCP_SERVER_HOST_PORT', '8001')}/mcp",
            },
            "bakery_marmot": {
                "transport": "http",
                "url": os.environ["MARMOT_HOST"].rstrip("/") + "/api/v1/mcp",
                "headers": {"X-API-Key": os.environ["MARMOT_API_KEY"]},
            },
            "bakery_dbhub": {
                "transport": "http",
                "url": f"http://localhost:{os.environ['DBHUB_HOST_PORT']}/mcp",
            },
        }
    )

    started = perf_counter()
    tools = await client.get_tools()
    print(f"Nạp tools xong sau {perf_counter() - started:.2f}s: "
          f"{[tool.name for tool in tools]}")

    skills = load_skill_registry()
    skill_instructions = "\n\n".join(
        f"## Skill: {name}\n{skill['instructions']}"
        for name, skill in skills.items()
    )
    print(f"Skills được nạp: {list(skills)}")

    today = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()

    system_prompt = (
        "You are a management reporting assistant for bakery stakeholders and managers. Answer in the user's language. "
        f"Today is {today.isoformat()} in Asia/Ho_Chi_Minh. "
        "Marmot is your authoritative business knowledge base for this bakery. "
        "Its catalog contains the documented business context: definitions, operating processes, "
        "business rules, source tables and columns, relationships, data grain, units and "
        "currencies, lineage and ownership. For business questions, consult the relevant "
        "Marmot knowledge to understand the question and establish its meaning before planning "
        "calculations or interpreting results. Reuse catalog knowledge already retrieved "
        "when it is sufficient. If information is missing or ambiguous, state the gap "
        "and make any assumptions explicit. "
        "Cube model inspection shows queryable cubes, measures and dimensions, "
        "not the business catalog. "
        "Prefer DBHub named reporting tools for covered management questions; supply values without writing SQL. Use get_reporting_coverage for current reports and empty periods. Never silently change dates or fill missing days with zeros. Use Cube semantic queries for other calculations expressible by those members. "
        "DBHub executes read-only SQL against Cube's SQL API for analysis needing "
        "CTEs, subqueries or other SQL composition. "
        "Choose the query interface that expresses the requested calculation accurately, "
        "follow the supplied skills for tool inputs, and report limitations or uncertainty."
    )

    agent = create_agent(
        model="openai:gpt-4o-mini-2024-07-18",
        # model="openai:gpt-6-luna",
        tools=tools,
        system_prompt=system_prompt + "\n\n# Skills\n\n" + skill_instructions,
    )

    request = {
        "messages": [
            {
                "role": "user",
                "content": user_question,
            }
        ]
    }

    final_answer = None
    previous_event = perf_counter()

    async for update in agent.astream(
        request,
        stream_mode="updates",
    ):
        now = perf_counter()
        step_seconds = now - previous_event
        previous_event = now

        for node, data in update.items():
            print(f"\n[{now - started:.2f}s, +{step_seconds:.2f}s] {node}")

            for message in data.get("messages", []):
                if message.type == "ai" and message.tool_calls:
                    for call in message.tool_calls:
                        print(f"  Gọi {call['name']}: {call['args']}")

                elif message.type == "tool":
                    content = message.content
                    if not isinstance(content, str):
                        content = json.dumps(content, ensure_ascii=False, default=str)
                    print(f"  {message.name} trả về: {content[:700]}")

                elif message.type == "ai" and message.content:
                    final_answer = message.content

    print(f"\nTổng thời gian: {perf_counter() - started:.2f}s")

    if final_answer:
        answer = final_answer

        if isinstance(answer, list):
            answer = "\n".join(
                part.get("text", "")
                for part in answer
                if isinstance(part, dict) and part.get("type") == "text"
            )

        display(Markdown(answer))
    else:
        print("Không nhận được trạng thái cuối từ agent.")


await main("""
For completed sales on October 7 and 8, 2026, using Asia/Ho_Chi_Minh dates, was the drop in total receipt amount driven more by fewer receipts or by a lower average receipt amount? Show both days’ receipt count, total amount and average receipt amount, with percentage changes. Then find the three products sold on both days with the largest decrease in units, showing each day’s units and the difference. Use the catalog’s definitions, explain the grain of each calculation, and state any limitation in the data.
""")